In [ ]:
import sys
if '/home/ec2-user/sagemaker-pipe-template' not in sys.path:
    sys.path.insert(0, '/home/ec2-user/sagemaker-pipe-template')
import boto3, pipeline.paths as paths, json

region='us-east-1'
model_package_group_name='abalone'
endpoint_name='abalone-endpoint'
endpoint_config_name='abalone-endpoint-config'
monitoring_schedule_names=['ab-dq']
model_version=1
data_bucket='omm-test-bucket'
boto_session=boto3.Session(region_name=region)
sm_client = boto_session.client('sagemaker', region_name=region)
p=paths.Paths(bucket_name='omm-test-bucket', project_name='abalone', model_prefix='abalone')

In [14]:
def pause_endpoint(endpoint_name):
    
    # 1. Get all monitoring schedules for this endpoint
    schedules = sm_client.list_monitoring_schedules(
        EndpointName=endpoint_name
    )
    
    # 2. Save schedule configs before deleting
    saved_schedules = []
    for schedule in schedules['MonitoringScheduleSummaries']:
        name = schedule['MonitoringScheduleName']
        
        # get full config
        details = sm_client.describe_monitoring_schedule(
            MonitoringScheduleName=name
        )
        saved_schedules.append({
            'MonitoringScheduleName': name,
            'MonitoringScheduleConfig': details['MonitoringScheduleConfig']
        })
        
        # delete schedule
        sm_client.delete_monitoring_schedule(MonitoringScheduleName=name)
        print(f"Deleted schedule: {name}")
    
    # 3. Save configs to S3 or local file for later restoration
    with open('monitoring_schedules_backup.json', 'w') as f:
        json.dump(saved_schedules, f, indent=2, default=str)
    print("Saved schedule configs to monitoring_schedules_backup.json")
    
    # 4. Now delete endpoint
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f"Deleted endpoint: {endpoint_name}")

pause_endpoint(endpoint_name)


Saved schedule configs to monitoring_schedules_backup.json
Deleted endpoint: abalone-endpoint


In [3]:
def resume_endpoint(endpoint_name, endpoint_config_name):
    
    # 1. Recreate endpoint
    sm_client.create_endpoint(
        EndpointName=endpoint_name,
        EndpointConfigName=endpoint_config_name
    )
    
    waiter = sm_client.get_waiter('endpoint_in_service')
    waiter.wait(EndpointName=endpoint_name)
    print(f"Endpoint ready: {endpoint_name}")
    
    # 2. Restore monitoring schedules from backup
    with open('monitoring_schedules_backup.json', 'r') as f:
        saved_schedules = json.load(f)
    
    for schedule in saved_schedules:
        sm_client.create_monitoring_schedule(
            MonitoringScheduleName=schedule['MonitoringScheduleName'],
            MonitoringScheduleConfig=schedule['MonitoringScheduleConfig']
        )
        print(f"Restored schedule: {schedule['MonitoringScheduleName']}")

resume_endpoint(endpoint_name, endpoint_config_name)

Endpoint ready: abalone-endpoint


In [ ]:
# Recreate later from the same config
sm_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name  # still exists
)

In [ ]:
# Del endpoint
sm_client.delete_endpoint(EndpointName='abalone-endpoint')

In [9]:
# Pause Monitors
for m in monitoring_schedule_names:
    sm_client.stop_monitoring_schedule(MonitoringScheduleName=m)

In [ ]:
# start_monitors
for m in monitoring_schedule_names:
    sm_client.start_monitoring_schedule(MonitoringScheduleName=m)

In [13]:
endpoint_name=endpoint_name
for job in sm_client.list_data_quality_job_definitions(EndpointName=endpoint_name)['JobDefinitionSummaries']:
    print(f"DELETING:{job['MonitoringJobDefinitionName']}")
    response = sm_client.delete_data_quality_job_definition(JobDefinitionName=job['MonitoringJobDefinitionName'])

for job in sm_client.list_model_bias_job_definitions(EndpointName=endpoint_name)['JobDefinitionSummaries']:
    print(f"DELETING:{job['MonitoringJobDefinitionName']}")
    response = sm_client.delete_model_bias_job_definition(JobDefinitionName=job['MonitoringJobDefinitionName'])

for job in sm_client.list_model_quality_job_definitions(EndpointName=endpoint_name)['JobDefinitionSummaries']:
    print(f"DELETING:{job['MonitoringJobDefinitionName']}")
    response = sm_client.delete_model_quality_job_definition(JobDefinitionName=job['MonitoringJobDefinitionName'])

for job in sm_client.list_model_explainability_job_definitions(EndpointName=endpoint_name)['JobDefinitionSummaries']:
    print(f"DELETING:{job['MonitoringJobDefinitionName']}")
    response = sm_client.delete_model_explainability_job_definition(JobDefinitionName=job['MonitoringJobDefinitionName'])

sm_client.list_data_quality_job_definitions(EndpointName=endpoint_name)['JobDefinitionSummaries']

[]

In [5]:
# Delete all model packages in the group
model_packages = sm_client.list_model_packages(ModelPackageGroupName=model_package_group_name)
for mp in model_packages['ModelPackageSummaryList']:
    print(f"Deleting model package: {mp['ModelPackageArn']}")
    sm_client.delete_model_package(
        ModelPackageName=mp['ModelPackageArn']
    )

Deleting model package: arn:aws:sagemaker:us-east-1:088461143167:model-package/abalone/5
Deleting model package: arn:aws:sagemaker:us-east-1:088461143167:model-package/abalone/4
Deleting model package: arn:aws:sagemaker:us-east-1:088461143167:model-package/abalone/3
Deleting model package: arn:aws:sagemaker:us-east-1:088461143167:model-package/abalone/2
Deleting model package: arn:aws:sagemaker:us-east-1:088461143167:model-package/abalone/1


In [ ]:
# Delete the model package group itself
sm_client.delete_model_package_group(ModelPackageGroupName=model_package_group_name)
print("Deleted model package group: abalone")

Deleted model package group: abalone


In [ ]:
# Delete endpoint first (must delete before config)
try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print("Deleted endpoint")
    # Wait for deletion
    waiter = sm_client.get_waiter('endpoint_deleted')
    waiter.wait(EndpointName=endpoint_name)
except sm_client.exceptions.ClientError:
    print("Endpoint does not exist")

In [ ]:
# Delete endpoint config
try:
    sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    print("Deleted endpoint config")
except sm_client.exceptions.ClientError:
    print("Endpoint config does not exist")

In [ ]:
# Delete all objects with a specific prefix (directory)
bucket = s3_resource.Bucket(data_bucket)
bucket.objects.filter(Prefix=f'{project_path}/data/batch-output/').delete()
bucket.objects.filter(Prefix=f'{project_path}/data/capture/').delete()
bucket.objects.filter(Prefix=f'{project_path}/data/ground-truth/').delete()
bucket.objects.filter(Prefix=f'{project_path}/data/monitors/').delete()

Deleting model package: arn:aws:sagemaker:us-east-1:088461143167:model-package/abalone/1
Deleted model package group: abalone
Endpoint does not exist
Endpoint config does not exist


[]

In [ ]:
# Delete all models
models = sm_client.list_models()
for m in models['Models']:
    print(f"Deleting model: {m['ModelName']}")
    sm_client.delete_model(ModelName=m['ModelName'])